In [ ]:
# --- Vishalam setup: find the code dataset among the inputs, unpack, install (no secrets) ---
import glob, os, shutil, subprocess, sys, tarfile, time
T0 = time.time()
def hours_left(limit=11.2):  # Kaggle stops sessions at 12 h; keep a margin for saving outputs
    return max(0.25, limit - (time.time() - T0) / 3600)
WORK, INPUT, CODE = "/kaggle/working", "/kaggle/input", "/tmp/vishalam"

def inputs_with(marker, hint=""):
    """Input directories that contain `marker` (a relative path) and whose path contains `hint`."""
    hits = glob.glob(f"{INPUT}/**/{marker}", recursive=True)
    dirs = [h[: -len(marker)].rstrip("/") for h in hits]
    return sorted(d for d in set(dirs) if hint in d)

if not os.path.isdir(CODE):
    tars = glob.glob(f"{INPUT}/**/vishalam_code.tar.gz", recursive=True)
    if tars:
        os.makedirs(CODE)
        with tarfile.open(tars[0]) as t:
            t.extractall(CODE, filter="data")
    else:  # Kaggle sometimes unpacks archives itself
        roots = [d for d in inputs_with("vishalam/tokenizer/bpe.py") if os.path.exists(f"{d}/requirements.txt")]
        if not roots:
            raise SystemExit("code dataset not attached - run tools/kaggle_driver.py push-code and re-push this kernel")
        shutil.copytree(roots[0], CODE)
os.chdir(CODE)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("ready:", open("VERSION").read().strip() if os.path.exists("VERSION") else "code unpacked")


In [ ]:
# --- GPU guard: stop before spending quota if Kaggle gave us the wrong accelerator ---
import torch
names = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print("GPUs:", names)
if not names or not all("T4" in n for n in names):
    raise SystemExit(f"Expected T4 GPUs, got {names}. Set Accelerator -> 'GPU T4 x2' and run again.")


In [ ]:
# --- Evaluate the agent (generated seen/unseen, hand-written, HumanEval, MBPP) ---
TOKENIZER = inputs_with("tokenizer.json", "vishalam-data-1")[0] + "/tokenizer.json"
m1 = inputs_with("m1/test_seen.jsonl")[0] + "/m1"
MODEL = inputs_with("sft/model.pt", "vishalam-sft-m1-125m-")[0] + "/sft/model.pt"
# a session that ran out of time keeps its finished sets; the -b kernel, given that output, runs the rest
prev = inputs_with("results.json", "vishalam-eval-m1-125m")
skip = f"--skip-done {prev[0]}/results.json" if prev else ""
get_ipython().system(f"python -m vishalam.eval.run --weights {MODEL} --tokenizer {TOKENIZER} "
                     f"--sets test_seen,test_unseen,handwritten,humaneval,mbpp --data {m1} {skip} "
                     f"--sandbox subprocess --max-hours {hours_left() - 0.3:.2f} --out {WORK}/results.json")
